# Chapter 02: Regular Polygons

Source orientation: printed pages 26-38; PDF pages 44-56. The source pages were rendered privately from the scanned PDF only to orient this original notebook. No textbook prose, page image, exercise text, or figure crop is reproduced here.

## Chapter Goal

This chapter treats regular polygons as two linked objects: a constructible division of a circle and a finite symmetry machine. We will translate straightedge-and-compass language into roots of unity, translate symmetry language into matrices and group tables, and use those computations to inspect kaleidoscopes and star polygons.

The guiding questions are concrete:

- Which regular `n`-gons can Euclidean tools construct, and what does the odd part of `n` have to do with Fermat primes?
- Why does arbitrary angle trisection sit just outside straightedge-and-compass construction?
- How do rotations, reflections, and products of reflections organize the symmetries of a regular polygon?
- How does the same rotation orbit produce both ordinary polygons and star polygons `{p/q}`?

## Computational Translation Guide

| Book idea | Computational model in this notebook | What we check |
| --- | --- | --- |
| cyclotomy | roots of unity on the unit circle and the Gauss-Wantzel condition | constructible orders up to a finite bound have odd squarefree Fermat-prime part |
| angle trisection | the triple-angle cubic for a third angle | the 60-degree trisection cubic has degree 3, not a power of 2 |
| isometry | distance-preserving affine matrix maps | two fixed points force either identity on the plane or reflection in their line |
| symmetry group | finite set of transformations closed under product and inverse | Cayley table closure, orders, and defining relations for `C_n` and `D_n` |
| two reflections | reflection matrices in two mirror lines | product angle is twice the mirror angle, and order matters |
| kaleidoscope | orbit of a point under a dihedral group generated by two mirrors | a wedge of angle `pi/n` produces `2n` images with alternating orientation |
| star polygon | cyclic orbit with step `q` on `p` vertices | `gcd(p,q)` predicts orbit count, density, angle, and area formula |


In [ ]:
from __future__ import annotations

from pathlib import Path
import csv
import json
import math
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Arc, Circle
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import sympy as sp
from IPython.display import Markdown, display

CHAPTER_NO = 2
HERE = Path.cwd().resolve()
BOOK_ROOT = None
for candidate in [HERE, *HERE.parents]:
    if (candidate / "00-book-index.ipynb").exists() and (candidate / "utils").exists():
        BOOK_ROOT = candidate
        break
if BOOK_ROOT is None:
    raise RuntimeError("Could not locate Introduction-to-Geometry book root")

sys.path.insert(0, str(BOOK_ROOT))
from utils import assert_artifacts, display_artifact, ensure_chapter_artifact_dirs

artifact_dirs = ensure_chapter_artifact_dirs(CHAPTER_NO, BOOK_ROOT)
FIG = artifact_dirs["figures"]
HTML = artifact_dirs["html"]
CHECKS = artifact_dirs["checks"]
TABLES = artifact_dirs["tables"]
DATA = artifact_dirs["data"]

# Clear earlier scaffold outputs inside this chapter subtree before regenerating concept-named artifacts.
for stale_path in [FIG / "concept_configuration.svg", FIG / "parameter_experiment.svg", TABLES / "artifact_manifest.csv"]:
    if stale_path.exists():
        stale_path.unlink()

generated_artifacts: list[Path] = []
check_payloads: dict[str, dict] = {}

plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 180,
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 9,
})

def rel(path: Path) -> str:
    return path.resolve().relative_to(BOOK_ROOT.resolve()).as_posix()

def record(path: Path) -> Path:
    if path not in generated_artifacts:
        generated_artifacts.append(path)
    return path

def savefig(fig: plt.Figure, path: Path) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, bbox_inches="tight")
    normalize_svg(path)
    plt.close(fig)
    return record(path)

def write_json(path: Path, payload: dict) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, sort_keys=True), encoding="utf-8")
    return record(path)

def write_csv(path: Path, rows: list[dict]) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as handle:
        if not rows:
            handle.write("")
        else:
            writer = csv.DictWriter(handle, fieldnames=list(rows[0].keys()))
            writer.writeheader()
            writer.writerows(rows)
    return record(path)

def write_html(fig: go.Figure, path: Path) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        path.unlink()
    fig.write_html(path, include_plotlyjs="cdn", full_html=True)
    return record(path)

def unit_vertices(p: int, *, radius: float = 1.0, phase: float = 0.0) -> np.ndarray:
    angles = phase + 2 * np.pi * np.arange(p) / p
    return radius * np.column_stack([np.cos(angles), np.sin(angles)])

def star_order(p: int, q: int) -> list[int]:
    seen = []
    k = 0
    while k not in seen:
        seen.append(k)
        k = (k + q) % p
    return seen

def reflection_matrix(angle: float) -> np.ndarray:
    c = math.cos(2 * angle)
    s = math.sin(2 * angle)
    return np.array([[c, s], [s, -c]])

def rotation_matrix(angle: float) -> np.ndarray:
    c = math.cos(angle)
    s = math.sin(angle)
    return np.array([[c, -s], [s, c]])

def setup_equal_axis(ax, limit=1.25):
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(-limit, limit)
    ax.set_ylim(-limit, limit)
    ax.axis("off")


def normalize_svg(path: Path) -> Path:
    if path.suffix.lower() == ".svg" and path.exists():
        text = path.read_text(encoding="utf-8")
        cleaned = "\n".join(line.rstrip() for line in text.splitlines()) + "\n"
        if cleaned != text:
            path.write_text(cleaned, encoding="utf-8")
    return path


## Compact Visual Storyboard

The chapter-specific visual plan is intentionally small enough to audit:

| Sequence | Concept | Representation | Artifact |
| --- | --- | --- | --- |
| 1 | cyclotomy and constructible orders | roots of unity plus an order strip | `cyclotomy-constructible-orders.svg` |
| 2 | angle trisection obstruction | triple-angle cubic with exact degree check | `trisection-cubic-obstruction.svg` |
| 3 | isometry with two fixed points | two distance circles meeting in a reflected pair | `isometry-fixed-points-reflection.svg` |
| 4 | cyclic and dihedral symmetry | pentagon axes, group order table, Cayley table | `dihedral-d5-symmetry-axes.svg` |
| 5 | product of two reflections | mirror lines and doubled rotation angle | `reflection-product-angle-doubling.svg` |
| 6 | kaleidoscope | wedge orbit under `D_n` with orientation signs | `kaleidoscope-dihedral-wedge.svg`, `kaleidoscope-wedge-orbits.html` |
| 7 | star polygons | `{p/q}` gallery, orbit table, Plotly explorer | `star-polygon-density-gallery.svg`, `star-polygon-orbit-explorer.html` |

Every item has an invariant attached: constructibility condition, polynomial degree, equal-distance preservation, group closure, matrix residual, orbit size, or `gcd`/density checks.


## 1. Cyclotomy and the Trisection Boundary

Cyclotomy asks for regular polygons that can be inscribed by Euclid's ruler-and-compass tools. In computational terms, the vertices are roots of unity. The constructibility question becomes: can the angle `2*pi/n` be obtained through field extensions of degree a power of two?

The Gauss-Wantzel condition gives a practical finite test: after removing powers of two, the remaining odd part must be a product of distinct Fermat primes. The figure below is not a construction recipe; it is a map of which circle divisions are allowed by that criterion.

Angle trisection enters as a warning. To trisect an arbitrary angle by setting `x = sin(theta/3)`, the triple-angle identity gives `4*x^3 - 3*x + sin(theta) = 0`. For a 60-degree angle this produces a cubic of degree 3. A straightedge-and-compass construction can only build numbers through repeated quadratic steps, so this cubic degree is the obstruction signal.


In [ ]:
FERMAT_PRIMES = [3, 5, 17, 257, 65537]

def constructible_order(n: int) -> bool:
    if n < 3:
        return False
    odd = n
    while odd % 2 == 0:
        odd //= 2
    for p in FERMAT_PRIMES:
        if odd % p == 0:
            odd //= p
            if odd % p == 0:
                return False
    return odd == 1

constructible_rows = []
for n in range(3, 81):
    odd = n
    while odd % 2 == 0:
        odd //= 2
    constructible_rows.append({
        "n": n,
        "constructible": constructible_order(n),
        "odd_part": odd,
        "central_angle_degrees": 360 / n,
    })
constructible_table = pd.DataFrame(constructible_rows)
constructible_csv = write_csv(TABLES / "constructible-orders.csv", constructible_rows)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), gridspec_kw={"width_ratios": [1.05, 1.45]})
ax = axes[0]
setup_equal_axis(ax, 1.18)
verts17 = unit_vertices(17, phase=np.pi / 2)
ax.add_patch(Circle((0, 0), 1, fill=False, lw=1.1, color="#59606d"))
ax.scatter(verts17[:, 0], verts17[:, 1], s=18, color="#1b6ca8", zorder=3)
for idx in [0, 1, 2, 4, 8, 16]:
    x0, y0 = verts17[idx]
    ax.plot([0, x0], [0, y0], lw=0.7, color="#9aa3ad")
    ax.text(1.08 * x0, 1.08 * y0, str(idx), ha="center", va="center", fontsize=8)
ax.plot(verts17[[0, 1], 0], verts17[[0, 1], 1], lw=2.3, color="#d1495b")
ax.set_title("17th roots of unity")
ax.text(0, -1.17, "Vertices are powers of one rotation", ha="center", va="top", fontsize=9)

ax = axes[1]
xs = np.arange(3, 41)
ys = np.zeros_like(xs)
colors = ["#2a9d8f" if constructible_order(int(n)) else "#c8cdd3" for n in xs]
ax.scatter(xs, ys, s=75, c=colors, edgecolors="#333333", linewidths=0.4, zorder=3)
for n in xs:
    if constructible_order(int(n)):
        ax.text(n, 0.13, str(n), ha="center", va="bottom", fontsize=8, rotation=90)
ax.set_ylim(-0.35, 0.55)
ax.set_yticks([])
ax.set_xlim(2, 41)
ax.set_xlabel("number of sides n")
ax.set_title("Constructible orders up to 40")
ax.grid(axis="x", color="#e6e8eb", linewidth=0.7)
ax.text(4.5, -0.23, "green = passes Fermat-prime test", fontsize=9, color="#2a6f64")
fig.suptitle("Cyclotomy as circle division plus an arithmetic filter", y=1.02)
cyclotomy_fig = savefig(fig, FIG / "cyclotomy-constructible-orders.svg")

x = sp.symbols("x")
trisection_poly = 4 * x**3 - 3 * x + sp.sqrt(3) / 2
minpoly_cos20 = sp.minpoly(2 * sp.cos(sp.pi / 9), x)
poly_degree = int(sp.degree(minpoly_cos20, x))

xs = np.linspace(-1.05, 1.05, 500)
ys = 4 * xs**3 - 3 * xs + math.sqrt(3) / 2
roots = [float(sp.N(root)) for root in sp.nroots(trisection_poly)]
fig, ax = plt.subplots(figsize=(7.8, 4.4))
ax.axhline(0, color="#30343b", lw=0.8)
ax.axvline(0, color="#d5d9df", lw=0.8)
ax.plot(xs, ys, color="#7b2cbf", lw=2.2, label=r"$4x^3 - 3x + \sin(60^\circ)$")
for root in roots:
    ax.scatter([root], [0], s=42, color="#f77f00", zorder=4)
    ax.text(root, 0.16, f"{root:.3f}", ha="center", fontsize=8)
ax.set_title("Trisection turns a familiar angle into a cubic")
ax.set_xlabel(r"candidate $x = \sin(\theta/3)$")
ax.set_ylabel("cubic residual")
ax.grid(color="#e7eaef", linewidth=0.7)
ax.legend(loc="upper left", frameon=False)
ax.text(-1.0, min(ys) + 0.3, f"degree of minpoly(2 cos 20 deg) = {poly_degree}", fontsize=9)
trisection_fig = savefig(fig, FIG / "trisection-cubic-obstruction.svg")

cyclotomy_checks = {
    "constructible_examples": constructible_table.query("constructible and n <= 40")["n"].astype(int).tolist(),
    "nonconstructible_examples": [7, 9, 11, 13, 14, 18],
    "n_17_constructible": constructible_order(17),
    "n_51_constructible": constructible_order(51),
    "n_7_constructible": constructible_order(7),
    "trisection_minpoly_2cos20": str(minpoly_cos20),
    "trisection_minpoly_degree": poly_degree,
    "cubic_root_residual_max": float(max(abs(4 * r**3 - 3 * r + math.sqrt(3) / 2) for r in roots)),
}
cyclotomy_check_path = write_json(CHECKS / "cyclotomy-trisection-checks.json", cyclotomy_checks)
check_payloads["cyclotomy_trisection"] = cyclotomy_checks

for path in [cyclotomy_fig, trisection_fig, constructible_csv, cyclotomy_check_path]:
    display(Markdown(f"**{rel(path)}**"))
    display_artifact(path, width=820)


## 2. Isometry: Fixed Points Leave Little Freedom

An isometry is a one-to-one transformation that preserves every distance. If two points `A` and `B` are fixed, then a third point `P` must go to a point with the same distance from `A` and the same distance from `B`. Geometrically, the possible image lies at the intersection of two circles. Except on the line `AB`, those circles meet in exactly two points: `P` itself and the mirror image of `P` across `AB`.

That is the computational content of the chapter's fixed-point claim. More than one fixed point forces the isometry to be either the identity or a reflection in the fixed line.


In [ ]:
A = np.array([-0.65, 0.0])
B = np.array([0.75, 0.0])
P = np.array([0.18, 0.78])
P_ref = P.copy(); P_ref[1] *= -1
rA = np.linalg.norm(P - A)
rB = np.linalg.norm(P - B)

fig, ax = plt.subplots(figsize=(7, 5))
setup_equal_axis(ax, 1.45)
for center, radius in [(A, rA), (B, rB)]:
    ax.add_patch(Circle(center, radius, fill=False, lw=1.4, linestyle="--", color="#8d99ae"))
ax.plot([A[0], B[0]], [A[1], B[1]], color="#30343b", lw=2.0)
ax.scatter([A[0], B[0]], [A[1], B[1]], color="#30343b", s=38, zorder=4)
ax.scatter([P[0], P_ref[0]], [P[1], P_ref[1]], color=["#e76f51", "#2a9d8f"], s=48, zorder=5)
ax.plot([P[0], P_ref[0]], [P[1], P_ref[1]], color="#adb5bd", lw=1.0)
ax.text(A[0], A[1]-0.08, "A", ha="center", va="top")
ax.text(B[0], B[1]-0.08, "B", ha="center", va="top")
ax.text(P[0]+0.05, P[1]+0.04, "P")
ax.text(P_ref[0]+0.05, P_ref[1]-0.08, "reflected P")
ax.text(0.0, 0.08, "fixed line AB", ha="center", color="#30343b")
ax.set_title("Two fixed points force identity or reflection")
isometry_fig = savefig(fig, FIG / "isometry-fixed-points-reflection.svg")

isometry_checks = {
    "distance_AP_equals_AP_reflected": float(abs(np.linalg.norm(P - A) - np.linalg.norm(P_ref - A))),
    "distance_BP_equals_BP_reflected": float(abs(np.linalg.norm(P - B) - np.linalg.norm(P_ref - B))),
    "fixed_line_y_coordinate": 0.0,
    "interpretation": "Only P and its mirror image across AB satisfy both distance equations.",
}
isometry_check_path = write_json(CHECKS / "isometry-fixed-point-checks.json", isometry_checks)
check_payloads["isometry_fixed_points"] = isometry_checks

for path in [isometry_fig, isometry_check_path]:
    display(Markdown(f"**{rel(path)}**"))
    display_artifact(path, width=760)


## 3. Cyclic and Dihedral Symmetry Groups

A regular `n`-gon has `n` rotations and `n` reflections. The rotations alone form the cyclic group `C_n`: applying the basic turn repeatedly cycles through all vertices and eventually returns to the identity. Adding one reflection gives the dihedral group `D_n`: the reflection reverses orientation and conjugates a rotation into its inverse.

The useful computational representation is a pair `(epsilon, k)`. If `epsilon = 0`, the operation is a rotation `r^k`. If `epsilon = 1`, the operation is a reflection `s r^k`. The product rule encodes non-commutativity: reflections flip the sign of the next rotation exponent.


In [ ]:
def d_elem_label(elem: tuple[int, int]) -> str:
    eps, k = elem
    if eps == 0:
        return "e" if k == 0 else f"r^{k}"
    return "s" if k == 0 else f"sr^{k}"

def d_compose(a: tuple[int, int], b: tuple[int, int], n: int) -> tuple[int, int]:
    eps_a, k_a = a
    eps_b, k_b = b
    return ((eps_a + eps_b) % 2, (k_a + ((-1) ** eps_a) * k_b) % n)

def d_inverse(a: tuple[int, int], n: int) -> tuple[int, int]:
    for b in [(eps, k) for eps in [0, 1] for k in range(n)]:
        if d_compose(a, b, n) == (0, 0) and d_compose(b, a, n) == (0, 0):
            return b
    raise ValueError(a)

n = 5
elements = [(0, k) for k in range(n)] + [(1, k) for k in range(n)]
cayley_rows = []
for a in elements:
    row = {"left_after_right": d_elem_label(a)}
    for b in elements:
        row[d_elem_label(b)] = d_elem_label(d_compose(a, b, n))
    cayley_rows.append(row)
cayley_csv = write_csv(TABLES / "dihedral-d5-cayley-table.csv", cayley_rows)

order_rows = []
for m in range(1, 9):
    order_rows.append({"group": f"C_{m}", "operations": m, "rotations": m, "reflections": 0, "presentation": f"r^{m}=e"})
    order_rows.append({"group": f"D_{m}", "operations": 2 * m, "rotations": m, "reflections": m, "presentation": f"s^2=e, r^{m}=e, srs=r^-1"})
order_csv = write_csv(TABLES / "symmetry-group-orders.csv", order_rows)

verts = unit_vertices(n, phase=np.pi / 2)
fig, ax = plt.subplots(figsize=(6.8, 6.2))
setup_equal_axis(ax, 1.35)
poly = np.vstack([verts, verts[0]])
ax.plot(poly[:, 0], poly[:, 1], color="#1f2937", lw=1.8)
ax.scatter(verts[:, 0], verts[:, 1], color="#1b6ca8", s=42, zorder=3)
for k, (xv, yv) in enumerate(verts):
    ax.text(1.11 * xv, 1.11 * yv, f"P{k}", ha="center", va="center", fontsize=9)
for k in range(n):
    angle = np.pi / 2 + k * np.pi / n
    line = np.array([[math.cos(angle), math.sin(angle)], [-math.cos(angle), -math.sin(angle)]])
    ax.plot(line[:, 0], line[:, 1], color="#8d99ae", lw=0.8, linestyle="--")
arc = Arc((0, 0), 0.72, 0.72, theta1=90, theta2=90 + 360 / n, color="#e76f51", lw=2.0)
ax.add_patch(arc)
ax.annotate("r", xy=(0.24, 0.30), xytext=(0.46, 0.48), arrowprops={"arrowstyle": "->", "color": "#e76f51"}, color="#e76f51")
ax.text(0, -1.26, "Dashed mirror axes plus rotations generate D5", ha="center", fontsize=10)
ax.set_title("Dihedral symmetry of a regular pentagon")
dihedral_fig = savefig(fig, FIG / "dihedral-d5-symmetry-axes.svg")

closure_ok = all(d_compose(a, b, n) in elements for a in elements for b in elements)
inverse_ok = all(d_compose(a, d_inverse(a, n), n) == (0, 0) for a in elements)
r = (0, 1)
s = (1, 0)
relations = {
    "r_power_n_is_identity": (0, n % n) == (0, 0),
    "s_squared_is_identity": d_compose(s, s, n) == (0, 0),
    "srs_equals_r_inverse": d_compose(d_compose(s, r, n), s, n) == d_inverse(r, n),
    "sr_not_rs": d_compose(s, r, n) != d_compose(r, s, n),
}
dihedral_checks = {"n": n, "element_count": len(elements), "closure_ok": closure_ok, "inverse_ok": inverse_ok, **relations}
dihedral_check_path = write_json(CHECKS / "dihedral-group-checks.json", dihedral_checks)
check_payloads["dihedral_group"] = dihedral_checks

for path in [dihedral_fig, order_csv, cayley_csv, dihedral_check_path]:
    display(Markdown(f"**{rel(path)}**"))
    display_artifact(path, width=760)


## 4. The Product of Two Reflections

The chapter's most productive mechanism is the product of reflections. Reflect across one mirror, then across a second mirror through the same point. The result is a rotation about the intersection point, and the rotation angle is twice the angle from the first mirror to the second.

The order matters. Reversing the mirrors gives the inverse rotation. This is the first place where the group product is visibly non-commutative.


In [ ]:
theta = math.pi / 5
R0 = reflection_matrix(0.0)
Rtheta = reflection_matrix(theta)
product = Rtheta @ R0
expected = rotation_matrix(2 * theta)
reverse = R0 @ Rtheta

point = np.array([0.82, 0.22])
p1 = R0 @ point
p2 = Rtheta @ p1
p_reverse = reverse @ point

fig, ax = plt.subplots(figsize=(7, 6))
setup_equal_axis(ax, 1.25)
for angle, name, color in [(0.0, "mirror 1", "#30343b"), (theta, "mirror 2", "#2a9d8f")]:
    line = np.array([[-math.cos(angle), -math.sin(angle)], [math.cos(angle), math.sin(angle)]])
    ax.plot(line[:, 0], line[:, 1], lw=2.0, color=color)
    ax.text(1.02 * math.cos(angle), 1.02 * math.sin(angle), name, color=color, fontsize=9)
ax.add_patch(Arc((0, 0), 0.42, 0.42, theta1=0, theta2=math.degrees(theta), color="#2a9d8f", lw=1.5))
ax.add_patch(Arc((0, 0), 0.68, 0.68, theta1=0, theta2=math.degrees(2 * theta), color="#e76f51", lw=2.0))
for pnt, label, color in [(point, "P", "#1b6ca8"), (p1, "after mirror 1", "#8d99ae"), (p2, "after both", "#e76f51"), (p_reverse, "reverse order", "#7b2cbf")]:
    ax.scatter([pnt[0]], [pnt[1]], s=46, color=color, zorder=4)
    ax.text(pnt[0] + 0.04, pnt[1] + 0.04, label, fontsize=8, color=color)
ax.plot([point[0], p1[0], p2[0]], [point[1], p1[1], p2[1]], color="#adb5bd", lw=1.0, linestyle="--")
ax.text(0.34, 0.09, "theta", color="#2a9d8f")
ax.text(0.47, 0.30, "2 theta", color="#e76f51")
ax.set_title("Two intersecting mirrors multiply to a rotation")
reflection_product_fig = savefig(fig, FIG / "reflection-product-angle-doubling.svg")

sample_rows = []
for denom in [3, 4, 5, 6, 8]:
    a = math.pi / denom
    residual = np.max(np.abs(reflection_matrix(a) @ reflection_matrix(0.0) - rotation_matrix(2 * a)))
    reverse_residual = np.max(np.abs(reflection_matrix(0.0) @ reflection_matrix(a) - rotation_matrix(-2 * a)))
    sample_rows.append({"mirror_angle_pi_over": denom, "rotation_angle_degrees": 360 / denom, "forward_residual": residual, "reverse_residual": reverse_residual})
reflection_csv = write_csv(TABLES / "reflection-product-samples.csv", sample_rows)
reflection_checks = {
    "theta_radians": theta,
    "matrix_residual": float(np.max(np.abs(product - expected))),
    "reverse_is_inverse_residual": float(np.max(np.abs(reverse - rotation_matrix(-2 * theta)))),
    "noncommuting_residual": float(np.max(np.abs(product - reverse))),
}
reflection_check_path = write_json(CHECKS / "reflection-product-invariants.json", reflection_checks)
check_payloads["reflection_product"] = reflection_checks

for path in [reflection_product_fig, reflection_csv, reflection_check_path]:
    display(Markdown(f"**{rel(path)}**"))
    display_artifact(path, width=760)


## 5. Kaleidoscope: Dihedral Groups in a Wedge

A kaleidoscope is a physical model of a dihedral group. Put two mirrors at angle `pi/n`. A point placed between them produces `2n` visible images: `n` orientation-preserving images and `n` orientation-reversing images. The two mirrors generate the group because their product is the basic rotation by `2*pi/n`.

The static figure uses `n = 7`. The HTML artifact lets you inspect the same orbit for several wedge angles.


In [ ]:
def dihedral_orbit_points(n: int, point: np.ndarray) -> list[dict]:
    rows = []
    zeta_angle = 2 * np.pi / n
    for k in range(n):
        rot = rotation_matrix(k * zeta_angle)
        rows.append({"kind": "rotation", "k": k, "point": rot @ point, "orientation": 1})
        rows.append({"kind": "reflection", "k": k, "point": rot @ (reflection_matrix(0.0) @ point), "orientation": -1})
    return rows

n_k = 7
base_point = np.array([0.55 * math.cos(math.pi / (2 * n_k)), 0.55 * math.sin(math.pi / (2 * n_k))])
orbit = dihedral_orbit_points(n_k, base_point)
fig, ax = plt.subplots(figsize=(7, 7))
setup_equal_axis(ax, 1.08)
ax.add_patch(Circle((0, 0), 0.55, fill=False, lw=0.7, color="#e1e5ea"))
for angle in [0, math.pi / n_k]:
    ax.plot([0, math.cos(angle)], [0, math.sin(angle)], color="#30343b", lw=2.0)
for row in orbit:
    pnt = row["point"]
    marker = "o" if row["orientation"] == 1 else "s"
    color = "#1b6ca8" if row["orientation"] == 1 else "#e76f51"
    ax.scatter([pnt[0]], [pnt[1]], s=44, marker=marker, color=color, zorder=4)
for k in range(n_k):
    angle = k * 2 * np.pi / n_k
    ax.plot([0, math.cos(angle)], [0, math.sin(angle)], color="#d6dbe1", lw=0.55, linestyle="--")
ax.add_patch(Arc((0, 0), 0.34, 0.34, theta1=0, theta2=180/n_k, color="#2a9d8f", lw=1.7))
ax.text(0.22, 0.035, "pi/n", color="#2a9d8f")
ax.set_title("Kaleidoscope wedge generates a dihedral orbit")
ax.text(0, -1.02, f"n = {n_k}: {n_k} rotations and {n_k} reflected images", ha="center", fontsize=10)
kaleidoscope_fig = savefig(fig, FIG / "kaleidoscope-dihedral-wedge.svg")

frames = []
initial_data = None
for n_frame in range(3, 10):
    point = np.array([0.6 * math.cos(math.pi / (2 * n_frame)), 0.6 * math.sin(math.pi / (2 * n_frame))])
    rows = dihedral_orbit_points(n_frame, point)
    x_rot = [row["point"][0] for row in rows if row["orientation"] == 1]
    y_rot = [row["point"][1] for row in rows if row["orientation"] == 1]
    x_ref = [row["point"][0] for row in rows if row["orientation"] == -1]
    y_ref = [row["point"][1] for row in rows if row["orientation"] == -1]
    data = [
        go.Scatter(x=x_rot, y=y_rot, mode="markers", marker={"size": 11, "color": "#1b6ca8"}, name="rotation images"),
        go.Scatter(x=x_ref, y=y_ref, mode="markers", marker={"size": 10, "color": "#e76f51", "symbol": "square"}, name="reflected images"),
    ]
    if initial_data is None:
        initial_data = data
    frames.append(go.Frame(data=data, name=str(n_frame), layout=go.Layout(title_text=f"Kaleidoscope orbit for n={n_frame}")))
html_fig = go.Figure(data=initial_data, frames=frames)
html_fig.update_layout(
    title="Kaleidoscope wedge orbit: choose n with the slider",
    xaxis={"scaleanchor": "y", "range": [-1.05, 1.05], "zeroline": False},
    yaxis={"range": [-1.05, 1.05], "zeroline": False},
    width=760,
    height=620,
    sliders=[{"steps": [{"method": "animate", "label": frame.name, "args": [[frame.name], {"mode": "immediate", "frame": {"duration": 0}, "transition": {"duration": 0}}]} for frame in frames], "currentvalue": {"prefix": "n = "}}],
)
kaleidoscope_html = write_html(html_fig, HTML / "kaleidoscope-wedge-orbits.html")

kaleidoscope_checks = {
    "n": n_k,
    "orbit_count": len(orbit),
    "expected_orbit_count": 2 * n_k,
    "orientation_preserving_count": sum(row["orientation"] == 1 for row in orbit),
    "orientation_reversing_count": sum(row["orientation"] == -1 for row in orbit),
    "generator_rotation_angle": 2 * math.pi / n_k,
}
kaleidoscope_check_path = write_json(CHECKS / "kaleidoscope-orbit-checks.json", kaleidoscope_checks)
check_payloads["kaleidoscope"] = kaleidoscope_checks

for path in [kaleidoscope_fig, kaleidoscope_html, kaleidoscope_check_path]:
    display(Markdown(f"**{rel(path)}**"))
    display_artifact(path, width=820)


## 6. Star Polygons and Density

The same cyclic orbit that lists the vertices of an ordinary polygon also lists star polygons. Choose `p` equally spaced vertices and connect each vertex to the vertex `q` steps away. If `gcd(p, q) = 1`, one orbit visits all `p` vertices before returning home; if the gcd is larger, the drawing splits into several smaller cycles.

The density is the step `q` when `0 < q < p/2`. A ray from the center that avoids vertices crosses `q` edges. The formulas from the isosceles triangle with center angle `2*pi*q/p` are checked below for circumradius `R = 1`:

- half-side `l = sin(pi*q/p)`
- inradius-like altitude `r = cos(pi*q/p)`
- signed area sum `p*l*r = (p/2) sin(2*pi*q/p)`
- vertex angle `(1 - 2q/p) pi`


In [ ]:
def star_segments(p: int, q: int, radius: float = 1.0, phase: float = np.pi / 2) -> tuple[np.ndarray, list[tuple[int, int]]]:
    verts = unit_vertices(p, radius=radius, phase=phase)
    segments = [(i, (i + q) % p) for i in range(p)]
    return verts, segments

star_pairs = [(5, 2), (8, 3), (10, 3), (12, 5)]
fig, axes = plt.subplots(1, len(star_pairs), figsize=(13, 3.8))
for ax, (p, q) in zip(axes, star_pairs):
    setup_equal_axis(ax, 1.22)
    verts, segments = star_segments(p, q)
    for i, j in segments:
        ax.plot([verts[i, 0], verts[j, 0]], [verts[i, 1], verts[j, 1]], color="#1f2937", lw=1.2)
    ax.scatter(verts[:, 0], verts[:, 1], s=22, color="#f77f00", edgecolors="#222222", linewidths=0.35, zorder=3)
    order = star_order(p, q)
    ax.set_title(f"{{{p}/{q}}}")
    ax.text(0, -1.16, f"orbit {len(order)} of {p}; density {q}", ha="center", fontsize=8)
fig.suptitle("Star polygons as step-q cyclic orbits", y=1.02)
star_gallery_fig = savefig(fig, FIG / "star-polygon-density-gallery.svg")

star_rows = []
for p in range(5, 17):
    for q in range(1, p // 2 + 1):
        if 2 * q == p:
            continue
        g = math.gcd(p, q)
        half_side = math.sin(math.pi * q / p)
        altitude = math.cos(math.pi * q / p)
        area_sum = p * half_side * altitude
        star_rows.append({
            "p": p,
            "q": q,
            "gcd": g,
            "components": g,
            "orbit_length": p // g,
            "density": q,
            "central_angle_degrees": 360 * q / p,
            "vertex_angle_degrees": 180 * (1 - 2 * q / p),
            "area_for_R_1": area_sum,
        })
star_csv = write_csv(TABLES / "star-polygon-orbit-table.csv", star_rows)

frames = []
initial_data = None
for p, q in [(5, 1), (5, 2), (7, 2), (8, 3), (9, 2), (10, 3), (12, 5), (14, 3)]:
    verts, segments = star_segments(p, q)
    xs = []
    ys = []
    for i, j in segments:
        xs.extend([verts[i, 0], verts[j, 0], None])
        ys.extend([verts[i, 1], verts[j, 1], None])
    data = [
        go.Scatter(x=xs, y=ys, mode="lines", line={"color": "#1f2937", "width": 2}, name="edges"),
        go.Scatter(x=verts[:, 0], y=verts[:, 1], mode="markers", marker={"size": 8, "color": "#f77f00"}, name="vertices"),
    ]
    if initial_data is None:
        initial_data = data
    frames.append(go.Frame(data=data, name=f"{p}/{q}", layout=go.Layout(title_text=f"Star polygon {{{p}/{q}}}: gcd={math.gcd(p,q)}, density={q}")))
star_html_fig = go.Figure(data=initial_data, frames=frames)
star_html_fig.update_layout(
    title="Star polygon orbit explorer",
    xaxis={"scaleanchor": "y", "range": [-1.15, 1.15], "zeroline": False},
    yaxis={"range": [-1.15, 1.15], "zeroline": False},
    width=760,
    height=620,
    sliders=[{"steps": [{"method": "animate", "label": frame.name, "args": [[frame.name], {"mode": "immediate", "frame": {"duration": 0}, "transition": {"duration": 0}}]} for frame in frames], "currentvalue": {"prefix": "{p/q} = "}}],
)
star_html = write_html(star_html_fig, HTML / "star-polygon-orbit-explorer.html")

star_checks = {
    "coprime_examples_visit_all_vertices": all(len(star_order(p, q)) == p for p, q in star_pairs),
    "gcd_failure_example_12_4_orbit_length": len(star_order(12, 4)),
    "gcd_failure_example_12_4_components": math.gcd(12, 4),
    "area_formula_max_residual": float(max(abs(row["area_for_R_1"] - 0.5 * row["p"] * math.sin(2 * math.pi * row["q"] / row["p"])) for row in star_rows)),
    "vertex_angle_positive_for_rows": all(row["vertex_angle_degrees"] > 0 for row in star_rows),
}
star_check_path = write_json(CHECKS / "star-polygon-gcd-density-checks.json", star_checks)
check_payloads["star_polygons"] = star_checks

for path in [star_gallery_fig, star_html, star_csv, star_check_path]:
    display(Markdown(f"**{rel(path)}**"))
    display_artifact(path, width=820)


## Applied Lab: Change the Step, Watch the Group Change

The lab below is meant to be edited. Change the pairs in `lab_pairs` and re-run the cell. The diagnostics report whether the step visits all vertices, how many separate cycles appear, and whether the density and angle formulas remain consistent.

This is the chapter in miniature: one rotation creates an orbit, number theory decides whether that orbit is single or split, and reflections add the mirror symmetries around the same vertex set.


In [ ]:
lab_pairs = [(9, 2), (9, 3), (11, 4), (12, 5), (15, 4)]
lab_rows = []
for p, q in lab_pairs:
    order = star_order(p, q)
    g = math.gcd(p, q)
    lab_rows.append({
        "symbol": f"{{{p}/{q}}}",
        "p": p,
        "q": q,
        "gcd": g,
        "single_cycle": g == 1,
        "orbit_length_from_simulation": len(order),
        "orbit_length_from_gcd": p // g,
        "components": g,
        "density_if_q_lt_half_p": q if 2 * q < p else None,
        "vertex_angle_degrees": 180 * (1 - 2 * q / p),
    })
lab_csv = write_csv(TABLES / "lab-star-pair-diagnostics.csv", lab_rows)
lab_df = pd.DataFrame(lab_rows)
display(lab_df)
display(Markdown(f"Saved diagnostics to **{rel(lab_csv)}**"))


## Final Sanity Checks and Takeaways

The last cell writes the chapter summary artifacts and then asserts the important invariants. These checks are intentionally redundant with the local prose: a reader can inspect the figures, and the notebook can still verify that the algebraic and numerical claims did not drift.

Takeaways:

- Regular polygons are circle orbits, but constructible regular polygons are special circle orbits filtered by arithmetic.
- The trisection obstruction appears as a cubic degree where straightedge-and-compass constructions require repeated quadratic steps.
- Reflections are not just examples of isometries; their products generate rotations and explain the algebra of dihedral groups.
- A kaleidoscope is a physical `D_n` generator: two mirrors at angle `pi/n` create the full `2n`-image orbit.
- Star polygons `{p/q}` keep the same roots of unity but change the step. The `gcd(p,q)` and density `q` organize their geometry.


In [ ]:
visual_paths = [p for p in generated_artifacts if p.suffix.lower() in {".svg", ".png", ".jpg", ".jpeg", ".html"}]

assert constructible_order(17) is True
assert constructible_order(7) is False
assert constructible_order(9) is False
assert poly_degree == 3
assert check_payloads["isometry_fixed_points"]["distance_AP_equals_AP_reflected"] < 1e-12
assert check_payloads["isometry_fixed_points"]["distance_BP_equals_BP_reflected"] < 1e-12
assert check_payloads["dihedral_group"]["closure_ok"] is True
assert check_payloads["dihedral_group"]["inverse_ok"] is True
assert check_payloads["dihedral_group"]["srs_equals_r_inverse"] is True
assert check_payloads["reflection_product"]["matrix_residual"] < 1e-12
assert check_payloads["reflection_product"]["reverse_is_inverse_residual"] < 1e-12
assert check_payloads["kaleidoscope"]["orbit_count"] == check_payloads["kaleidoscope"]["expected_orbit_count"]
assert check_payloads["star_polygons"]["coprime_examples_visit_all_vertices"] is True
assert check_payloads["star_polygons"]["gcd_failure_example_12_4_orbit_length"] == 3
assert check_payloads["star_polygons"]["gcd_failure_example_12_4_components"] == 4
assert check_payloads["star_polygons"]["area_formula_max_residual"] < 1e-12

artifact_manifest_rows = []
for path in sorted(generated_artifacts, key=lambda item: rel(item)):
    artifact_manifest_rows.append({"path": rel(path), "kind": path.parent.name, "bytes": path.stat().st_size})
manifest_csv = write_csv(TABLES / "artifact-manifest.csv", artifact_manifest_rows)

visual_summary = {
    "chapter": CHAPTER_NO,
    "title": "Regular Polygons",
    "source_span": {"printed_pages": "26-38", "pdf_pages": "44-56"},
    "figures": [rel(path) for path in visual_paths if path.parent.name == "figures"],
    "html": [rel(path) for path in visual_paths if path.parent.name == "html"],
    "checks": sorted(check_payloads.keys()),
    "visual_count": len(visual_paths),
}
visual_summary_path = write_json(CHECKS / "visual_summary.json", visual_summary)

final_sanity = {
    "chapter": CHAPTER_NO,
    "artifact_count": len(generated_artifacts),
    "visual_count": len(visual_paths),
    "minimum_artifact_bytes": min(path.stat().st_size for path in generated_artifacts),
    "constructible_17": constructible_order(17),
    "nonconstructible_7": not constructible_order(7),
    "trisection_degree": poly_degree,
    "reflection_product_residual": check_payloads["reflection_product"]["matrix_residual"],
    "dihedral_d5_order": check_payloads["dihedral_group"]["element_count"],
    "kaleidoscope_orbit_count": check_payloads["kaleidoscope"]["orbit_count"],
    "star_area_residual": check_payloads["star_polygons"]["area_formula_max_residual"],
    "manifest": rel(manifest_csv),
    "visual_summary": rel(visual_summary_path),
}
final_sanity_path = write_json(CHECKS / "final-sanity-summary.json", final_sanity)

assert_artifacts(generated_artifacts + [manifest_csv, visual_summary_path, final_sanity_path], min_bytes=100)
assert all(path.stat().st_size > 20 for path in generated_artifacts)
assert len([path for path in visual_paths if path.parent.name == "figures"]) >= 5
assert len([path for path in visual_paths if path.parent.name == "html"]) >= 2
assert not (FIG / "concept_configuration.svg").exists()
assert not (FIG / "parameter_experiment.svg").exists()
assert not (TABLES / "artifact_manifest.csv").exists()

pd.DataFrame(artifact_manifest_rows).sort_values("path")
